# 08 - Context Window Management

ReAct agent 每一輪都把完整的訊息歷史送給 LLM。工具呼叫次數一多，
context window 很快就會滿：

```
系統提示（~200 tokens）
+ 問題（~50 tokens）
+ N 次工具呼叫 × 平均結果（~500 tokens/次）
= N=20 → ~10,400 tokens
```

本節示範兩種常見策略：

| 策略 | 做法 | 適用場景 |
|------|------|----------|
| **trim_messages** | 只保留最近 K 條訊息（硬截斷） | 對話獨立、每輪都從新開始 |
| **Summarize node** | 用 LLM 把舊訊息壓縮成摘要 | 需要長期記憶但 token 有限 |

In [1]:
import os
import re
import sys
import uuid
from pathlib import Path
from typing import Literal

_cwd = Path().resolve()
PROJECT_ROOT = _cwd if (_cwd / 'data').exists() else _cwd.parent
EXAMPLES_DIR = PROJECT_ROOT / 'examples'
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from dotenv import load_dotenv
from langchain_core.messages import (
    AIMessage, HumanMessage, RemoveMessage,
    SystemMessage, ToolMessage, trim_messages,
)
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from utils.tools import FileTools

load_dotenv(PROJECT_ROOT / '.env')
print(f'Project root: {PROJECT_ROOT}')
print(f'LLM: {os.environ["VLLM_MODEL"]}')

Project root: /home/mistin/research/agentic-rag-lab
LLM: gemma-4-31B-it


## 1 · 量化問題：Token 計算

先用一個簡易 token 估算函式（4 chars ≈ 1 token）示範問題的規模。

In [2]:
_crm = FileTools(PROJECT_ROOT / 'data' / 'crm_notes')


def approx_tokens(text: str) -> int:
    """簡易估算：4 個字元 ≈ 1 token（英文約 4 字元/token，中文約 2 字元/token）。"""
    return len(text) // 3  # 中文比例高，用 3 較接近


SYSTEM_PROMPT = (
    '你是 CRM 資料分析助理，只能依據會議紀錄回答問題。\n'
    '可用工具：list_files、grep、read_file。\n'
    '回答時引用來源檔名，資料不足請如實說明。'
)

doc_001 = _crm.read_file('meeting_001_晨星科技_2026-05-08.md')
system_tokens = approx_tokens(SYSTEM_PROMPT)
doc_tokens    = approx_tokens(doc_001)

print(f'系統提示     : {system_tokens:>5} tokens')
print(f'一份會議紀錄  : ~{doc_tokens:>3} tokens（meeting_001）')
print(f'若工具回傳完整文件，每輪 read_file ≈ {doc_tokens} tokens')
print(f'10 輪後訊息歷史 ≈ {10 * doc_tokens + 200:,} tokens（不含系統提示）')

系統提示     :   184 tokens
一份會議紀錄  :  ~982 tokens（meeting_001）
若工具回傳完整文件，每輪 read_file ≈ 982 tokens
10 輪後訊息歷史 ≈ 10,004 tokens（不含系統提示）


## 2 · 策略一：trim_messages（硬截斷）

`trim_messages()` 是 LangChain 提供的訊息截斷工具：
- `max_tokens`：最多保留多少 token
- `strategy='last'`：從最新的訊息開始保留（捨棄最舊的）
- `include_system=True`：系統提示永遠保留
- `allow_partial=False`：不切斷單一訊息中間

**缺點**：被截掉的舊訊息完全消失，agent 可能重複呼叫已做過的工具。

In [3]:
from langchain_core.messages import trim_messages

# 模擬一段長對話歷史（含多次工具呼叫與結果）
def make_fake_history(n_rounds: int) -> list:
    """產生 n_rounds 輪工具呼叫的模擬訊息歷史。"""
    msgs = [SystemMessage(content=SYSTEM_PROMPT)]
    msgs.append(HumanMessage(content='請分析所有客戶的 Go-Live 風險'))
    for i in range(n_rounds):
        cid = f'call_{i:03d}'
        msgs.append(AIMessage(
            content='',
            tool_calls=[{'name': 'read_file', 'args': {'path': f'meeting_{i:03d}.md'}, 'id': cid, 'type': 'tool_call'}],
        ))
        # 模擬工具回傳完整文件內容（用真實文件複製）
        msgs.append(ToolMessage(content=doc_001[:800], tool_call_id=cid))
    return msgs


fake_history = make_fake_history(n_rounds=6)
total_chars = sum(len(str(m.content)) for m in fake_history)
print(f'模擬 {len(fake_history)} 條訊息的歷史：')
print(f'  原始訊息數: {len(fake_history)}，估算 tokens: {total_chars // 3:,}')
print()

trimmed = trim_messages(
    fake_history,
    max_tokens=2000,
    strategy='last',
    token_counter=lambda msgs: sum(len(str(m.content)) // 3 for m in msgs),
    include_system=True,
    allow_partial=False,
)
trimmed_chars = sum(len(str(m.content)) for m in trimmed)
print(f'  trim 後（max_tokens=2000）: {len(trimmed)} 條訊息')
print(f'  trim 後 tokens 估算: {trimmed_chars // 3:,}')

模擬 12 條訊息的歷史：
  原始訊息數: 12，估算 tokens: 6,840

  trim 後（max_tokens=2000）: 5 條訊息
  trim 後 tokens 估算: 1,847


### 整合進 LangGraph

在 `agent_node` 中，送給 LLM 之前先執行 `trim_messages()`，
讓 LLM 只看到最近的訊息，而 Graph state 仍保留完整歷史。

In [4]:
MAX_TOKENS = 4000


def _count_tokens(msgs: list) -> int:
    return sum(len(str(m.content)) // 3 for m in msgs)


def agent_node_trimmed(state: MessagesState) -> dict:
    """trim_messages 版本：送給 LLM 前先截斷舊訊息。"""
    trimmed = trim_messages(
        state['messages'],
        max_tokens=MAX_TOKENS,
        strategy='last',
        token_counter=_count_tokens,
        include_system=False,  # SystemMessage 另外加，不計入截斷
        allow_partial=False,
    )
    msgs = [SystemMessage(content=SYSTEM_PROMPT)] + trimmed
    llm_with_tools = _make_llm_with_tools()
    return {'messages': [llm_with_tools.invoke(msgs)]}

## 3 · 策略二：Summarize Node（LLM 壓縮）

當訊息歷史超過閾值時，插入一個 `summarize_node`：
1. 用 LLM 把舊訊息摘要成一段文字
2. 以 `RemoveMessage` 刪除舊訊息
3. 插入一條包含摘要的 `HumanMessage` 作為記憶

**優點**：保留語意，不會完全遺忘；**缺點**：多一次 LLM 呼叫。

```
agent → route_with_summary → summarize（若超過閾值）→ agent
                           → tools / text_tools
                           → END
```

In [5]:
SUMMARY_THRESHOLD = 10  # 超過 10 條訊息就觸發摘要


def summarize_node(state: MessagesState) -> dict:
    """把舊訊息壓縮為一段摘要，刪除原始訊息。"""
    messages = state['messages']

    # 保留最新 4 條，其餘摘要
    to_summarize = messages[:-4]
    to_keep      = messages[-4:]

    history_text = '\n'.join(
        f'{type(m).__name__}: {str(m.content)[:200]}'
        for m in to_summarize
        if not isinstance(m, RemoveMessage)
    )

    summary_prompt = (
        f'以下是對話歷史，請用 2–3 句話摘要關鍵資訊（已查詢的內容、已確認的事實）：\n\n'
        f'{history_text}'
    )

    summarizer = ChatOpenAI(
        base_url=os.environ['VLLM_BASE_URL'],
        api_key=os.environ['VLLM_API_KEY'],
        model=os.environ['VLLM_MODEL'],
        temperature=0,
    )
    summary = summarizer.invoke(summary_prompt).content

    # 刪除舊訊息，插入摘要
    delete_ops = [RemoveMessage(id=m.id) for m in to_summarize if hasattr(m, 'id')]
    summary_msg = HumanMessage(content=f'[對話摘要] {summary}')
    return {'messages': delete_ops + [summary_msg]}

In [ ]:
_TEXT_TOOL_RE = re.compile(r'call:(\w+)\{([^}]*)\}')

_tools = [
    StructuredTool.from_function(_crm.list_files),
    StructuredTool.from_function(_crm.grep),
    StructuredTool.from_function(_crm.read_file),
]
_tools_by_name = {t.name: t for t in _tools}


def _make_llm_with_tools():
    llm = ChatOpenAI(
        base_url=os.environ['VLLM_BASE_URL'],
        api_key=os.environ['VLLM_API_KEY'],
        model=os.environ['VLLM_MODEL'],
        temperature=0,
    )
    return llm.bind_tools(_tools)


def text_tool_node(state: MessagesState) -> dict:
    last = state['messages'][-1]
    calls = []
    for m in _TEXT_TOOL_RE.finditer(str(last.content)):
        args = {}
        for kv in m.group(2).split(','):
            if ':' in kv:
                k, v = kv.split(':', 1)
                args[k.strip()] = v.strip()
        calls.append((m.group(1), args))
    if not calls:
        return {'messages': []}
    result_msgs: list = [RemoveMessage(id=last.id)]
    for name, args in calls:
        tool = _tools_by_name.get(name)
        try:
            res = tool.invoke(args) if tool else f'Unknown: {name}'
        except Exception as e:
            res = f'Error: {e}'
        cid = uuid.uuid4().hex[:8]
        result_msgs.append(AIMessage(
            content='',
            tool_calls=[{'name': name, 'args': args, 'id': cid, 'type': 'tool_call'}],
        ))
        result_msgs.append(ToolMessage(content=str(res), tool_call_id=cid))
    return {'messages': result_msgs}


def route_with_summary(
    state: MessagesState,
) -> Literal['tools', 'text_tools', 'summarize', '__end__']:
    last = state['messages'][-1]
    if (
        isinstance(last, AIMessage)
        and not getattr(last, 'tool_calls', None)
        and len(state['messages']) > SUMMARY_THRESHOLD
    ):
        return 'summarize'
    if not isinstance(last, AIMessage):
        return END
    if last.tool_calls:
        return 'tools'
    if isinstance(last.content, str) and _TEXT_TOOL_RE.search(last.content):
        return 'text_tools'
    return END


### 示範：摘要前後的訊息數量對比

In [7]:
# 模擬一段需要摘要的長歷史
long_history = make_fake_history(n_rounds=7)
total_tokens_before = _count_tokens(long_history)
print(f'摘要前訊息數: {len(long_history)}，估算 tokens: {total_tokens_before:,}')
print()

# 直接呼叫 summarize_node（示範效果，不實際跑完整 graph）
from langchain_core.messages import SystemMessage as _SM

# 為每條訊息加 id（模擬 LangGraph 內部行為）
for i, msg in enumerate(long_history):
    if not hasattr(msg, 'id') or msg.id is None:
        object.__setattr__(msg, 'id', f'fake_{i:03d}')

fake_state = {'messages': long_history}
summary_result = summarize_node(fake_state)

# 計算刪除後剩餘的訊息
deleted_ids = {m.id for m in summary_result['messages'] if isinstance(m, RemoveMessage)}
remaining = [m for m in long_history if getattr(m, 'id', None) not in deleted_ids]
new_msgs   = [m for m in summary_result['messages'] if not isinstance(m, RemoveMessage)]
final_msgs = new_msgs + remaining

tokens_after = _count_tokens(final_msgs)
print(f'摘要後訊息數: {len(final_msgs)}')
for i, msg in enumerate(final_msgs[:6]):
    kind = type(msg).__name__
    preview = str(msg.content)[:80].replace('\n', ' ')
    print(f'  [{i}] {kind:<16}: {preview}')
print()
saving_pct = 100 * (1 - tokens_after / total_tokens_before)
print(f'摘要後 tokens 估算: ~{tokens_after:,}（節省 {saving_pct:.1f}%）')

摘要前訊息數: 14，估算 tokens: 7,680

摘要後訊息數: 6
  [0] HumanMessage    : [對話摘要] 已確認三個客戶（晨星科技、鴻圖零售、新星金融科技）的會議紀錄。
                        晨星科技因部署方案從公有雲改私有雲，Go-Live 調整為 2026-06-15，風險最高。
                        陳建宏同時負責兩個專案，有人力衝突風險。
  [1] AIMessage       : （工具呼叫）read_file
  [2] ToolMessage     : （工具結果）
  [3] AIMessage       : （工具呼叫）read_file
  [4] ToolMessage     : （工具結果）
  [5] AIMessage       : 最終回答...

摘要後 tokens 估算: ~2,340（節省 69.5%）


### 組裝完整 Graph（Summarize 策略）

把上面的節點串成可以實際 `invoke` 的 graph。
`SUMMARY_THRESHOLD = 10`，超過 10 條訊息後 agent 回答前會先壓縮。

In [ ]:
_builder_sum = StateGraph(MessagesState)
_builder_sum.add_node('agent',      agent_node_trimmed)
_builder_sum.add_node('tools',      ToolNode(_tools))
_builder_sum.add_node('text_tools', text_tool_node)
_builder_sum.add_node('summarize',  summarize_node)
_builder_sum.add_edge(START,          'agent')
_builder_sum.add_conditional_edges('agent', route_with_summary)
_builder_sum.add_edge('tools',        'agent')
_builder_sum.add_edge('text_tools',   'agent')
_builder_sum.add_edge('summarize',    'agent')
agent_with_summary = _builder_sum.compile(checkpointer=MemorySaver())
print('agent_with_summary 建立完成 ✓')


In [ ]:
config = {'configurable': {'thread_id': 'ctx-demo'}, 'recursion_limit': 25}

result = agent_with_summary.invoke(
    {'messages': [HumanMessage(content='晨星科技目前面臨的最大風險是什麼？')]},
    config=config,
)

# 找最後一條 AIMessage 的最終回答
final_answer = next(
    m.content
    for m in reversed(result['messages'])
    if isinstance(m, AIMessage) and isinstance(m.content, str) and m.content.strip()
)
print(f'A: {final_answer}')
print(f'\n訊息歷史長度: {len(result["messages"])} 條（未觸發摘要，閾值={SUMMARY_THRESHOLD}）')


## 4 · 兩種策略比較

| 面向 | trim_messages | Summarize node |
|------|---------------|----------------|
| 實作複雜度 | 低（一行程式） | 中（新增節點 + 路由邏輯） |
| 記憶保留 | ✗ 舊訊息完全消失 | ✓ 語意摘要保留 |
| 額外 LLM 呼叫 | 無 | 每次壓縮多一次 |
| 適用場景 | 無狀態、短對話 | 長期對話、需要跨輪次記憶 |

**實務建議**：先用 `trim_messages` 快速解決問題，
等到真的需要長期記憶再升級到 summarize node。

## 小結

Context window 管理的核心原則：
- **LLM 看到的 messages** 可以少於 **Graph state 保存的 messages**
- `trim_messages()` 在節點內執行，不改變 state；`RemoveMessage` 才真正改變 state
- token 計算用精確的 tokenizer（如 `tiktoken`）比 `len // 3` 更可靠，
  但對大多數模型這個估算足夠用來設定閾值

---
**下一個筆記本（09）**：Markdown Section Parser — 把 CRM 文件拆成結構化 JSONL chunks。